# Adaptive Video Learning System - 4 Dataset Training (WORKING VERSION)
Fine-tuning Mistral 7B on SciQ, GSM8K, OpenBookQA and ELI5

**Features:**
- Multiple questions per assessment
- Aggregate evaluation across all answers  
- Recommendations based on user preference (video vs article)

In [ ]:
# Install packages
!pip install -q datasets transformers accelerate peft bitsandbytes trl torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 47.7 MB/s eta 0:00:00


In [ ]:
# Setup Kaggle credentials
import os
os.environ['KAGGLE_USERNAME'] = 'yahia saqer'
os.environ['KAGGLE_KEY'] = 'KGAT_9502dfc610067da2128caa08ad564633'

# Install and configure Kaggle
!pip install -q kaggle

# Download ELI5 dataset
print("Downloading ELI5 dataset (6GB, this will take a few minutes)...")
!kaggle datasets download -d trandaiphu/eli5-dataset
!unzip -q eli5-dataset.zip

print("ELI5 downloaded!")

Dataset URL: https://www.kaggle.com/datasets/trandaiphu/eli5-dataset
License(s): unknown
 99% 1.94G/1.96G [00:01<00:00, 1.09GB/s]
100% 1.96G/1.96G [00:01<00:00, 1.16GB/s]
ELI5 downloaded!


In [ ]:
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer
import random
from typing import List, Dict

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
!nvidia-smi

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Wed Jan 28 08:51:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             43W /  400W |       5MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------

## Load Datasets

In [ ]:
print("Loading datasets...\n")

# 1. SciQ
sciq = load_dataset("allenai/sciq")
print(f"✓ SciQ: {len(sciq['train'])} train")

# 2. GSM8K
gsm8k = load_dataset("openai/gsm8k", "main")
print(f"✓ GSM8K: {len(gsm8k['train'])} train")

# 3. OpenBookQA
openbookqa = load_dataset("allenai/openbookqa", "main")
print(f"✓ OpenBookQA: {len(openbookqa['train'])} train")

# 4. ELI5 from Kaggle
import pandas as pd
import json

# Instead of loading the entire JSONL file, load a limited number of examples
eli5_data_list = []
limit = 50000  # Adjust this limit as needed to fit memory
print(f"Loading a sample of {limit} ELI5 examples...")
with open('ELI5.jsonl', 'r') as f:
    for i, line in enumerate(f):
        if i >= limit:
            break
        eli5_data_list.append(json.loads(line))

eli5_df = pd.DataFrame(eli5_data_list)
print(f"✓ ELI5: {len(eli5_df)} examples (sample)")

# Convert ELI5 to dict format for processing
eli5_data = eli5_df.to_dict('records')

print("\n" + "="*80)
print("All 4 datasets loaded!")
print("="*80)

Loading datasets...

✓ SciQ: 11679 train
✓ GSM8K: 7473 train
✓ OpenBookQA: 4957 train
Loading a sample of 50000 ELI5 examples...
✓ ELI5: 50000 examples (sample)

All 4 datasets loaded!


## Prompt Functions

In [ ]:
def create_multi_question_prompt_sciq(examples: List[Dict], correct: bool, preference: str) -> str:
    context = examples[0]['support']
    qa_section = ""

    for i, ex in enumerate(examples, 1):
        question = ex['question']
        if correct:
            answer = ex['correct_answer']
        else:
            distractors = [ex['distractor1'], ex['distractor2'], ex['distractor3']]
            answer = random.choice([d for d in distractors if d]) or ex['correct_answer']
        qa_section += f"Q{i}: {question}\nStudent Answer: {answer}\n\n"

    if correct:
        evaluation = f"""The student demonstrated strong understanding across all {len(examples)} questions, correctly answering each one and showing mastery of the concepts."""
        recommendation = f"""Ready to progress. Recommend: {'Next video' if preference == 'videos' else 'Next article'} on advanced topics."""
    else:
        evaluation = f"""The student shows misconceptions across {len(examples)} questions, indicating they need to review foundational concepts."""
        recommendation = f"""Needs reinforcement. Recommend: {'Review video' if preference == 'videos' else 'Review article'} on fundamentals."""

    return f"""### Domain: [SCIENCE]
### Context:
{context}

### Assessment:
{qa_section}
### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""


def create_multi_question_prompt_gsm8k(examples: List[Dict], correct: bool, preference: str) -> str:
    context = "Mathematics lesson on word problem solving."
    qa_section = ""

    for i, ex in enumerate(examples, 1):
        question = ex['question']
        if correct:
            full_answer = ex['answer']
            if '####' in full_answer:
                reasoning = full_answer.split('####')[0].strip()
                final = full_answer.split('####')[1].strip()
                answer = f"{reasoning[:100]}... Answer: {final}"
            else:
                answer = full_answer[:100]
        else:
            if '####' in ex['answer']:
                correct_num = ex['answer'].split('####')[1].strip()
                try:
                    wrong = str(int(float(correct_num)) + random.choice([-5, -2, 2, 5]))
                    answer = f"Incorrect approach. Answer: {wrong}"
                except:
                    answer = "Incorrect approach."
            else:
                answer = "Incorrect approach."
        qa_section += f"Q{i}: {question}\nStudent Answer: {answer}\n\n"

    if correct:
        evaluation = f"""The student correctly solved all {len(examples)} problems with proper reasoning and accurate answers."""
        recommendation = f"""Ready for harder problems. Recommend: {'Next video' if preference == 'videos' else 'Next article'} with advanced math."""
    else:
        evaluation = f"""The student struggled with {len(examples)} problems, making calculation errors or using wrong approaches."""
        recommendation = f"""Needs practice. Recommend: {'Review video' if preference == 'videos' else 'Review article'} on problem-solving basics."""

    return f"""### Domain: [MATH]
### Context:
{context}

### Assessment:
{qa_section}
### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""


def create_multi_question_prompt_openbook(examples: List[Dict], correct: bool, preference: str) -> str:
    context = examples[0].get('fact1', 'General knowledge and reasoning.')
    qa_section = ""

    for i, ex in enumerate(examples, 1):
        question = ex['question_stem']
        choices = ex['choices']
        if correct:
            answer_idx = ord(ex['answerKey']) - ord('A')
            answer = choices['text'][answer_idx]
        else:
            wrong_indices = [j for j in range(len(choices['text'])) if j != (ord(ex['answerKey']) - ord('A'))]
            answer = choices['text'][random.choice(wrong_indices)]
        qa_section += f"Q{i}: {question}\nStudent Answer: {answer}\n\n"

    if correct:
        evaluation = f"""The student answered all {len(examples)} reasoning questions correctly, showing strong critical thinking."""
        recommendation = f"""Ready for complex problems. Recommend: {'Next video' if preference == 'videos' else 'Next article'} with harder reasoning."""
    else:
        evaluation = f"""The student's incorrect answers suggest difficulty with logical reasoning."""
        recommendation = f"""Needs practice. Recommend: {'Review video' if preference == 'videos' else 'Review article'} on reasoning skills."""

    return f"""### Domain: [REASONING]
### Context:
{context}

### Assessment:
{qa_section}
### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""

def create_multi_question_prompt_eli5(example: Dict, correct: bool, preference: str) -> str:
    """ELI5 format with actual column names"""
    question = example['question']
    context = "Educational content explaining complex concepts."

    if correct:
        # Use the actual answer - it's likely a list or string
        answer_data = example['answers']
        if isinstance(answer_data, list):
            answer = answer_data[0][:200] if answer_data else "No answer"
        else:
            answer = str(answer_data)[:200]
    else:
        answer = "Incomplete or unclear explanation"

    if correct:
        evaluation = """The student provided a clear explanation showing understanding."""
        recommendation = f"""Ready for more. Recommend: {'Next video' if preference == 'videos' else 'Next article'} on related concepts."""
    else:
        evaluation = """The student's explanation shows gaps in understanding."""
        recommendation = f"""Needs review. Recommend: {'Review video' if preference == 'videos' else 'Review article'} on fundamentals."""

    return f"""### Domain: [GENERAL]
### Context:
{context}

### Assessment:
Q1: {question}
Student Answer: {answer}

### User Preference: {preference}

### Evaluation:
{evaluation}

### Recommendation:
{recommendation}"""

# Test
print("Sample prompts:")
print(create_multi_question_prompt_sciq([sciq['train'][0], sciq['train'][1]], True, "videos")[:500])

Sample prompts:
### Domain: [SCIENCE]
### Context:
Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic organisms have important uses in food preparation, including cheese, yogurt, beer and wine.

### Assessment:
Q1: What type of organism is commonly used in preparati


## Prepare Combined Dataset

In [ ]:
def prepare_combined_dataset(sciq_data, gsm8k_data, openbook_data, eli5_dataset_slice, split="train"):
    all_prompts = []
    preferences = ["videos", "articles"]

    # SciQ - group 3 questions
    print("Processing SciQ...")
    sciq_split = sciq_data[split]
    for i in range(0, len(sciq_split) - 2, 3):
        examples = [sciq_split[j] for j in range(i, min(i+3, len(sciq_split)))]
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_sciq(examples, True, pref))
        all_prompts.append(create_multi_question_prompt_sciq(examples, False, random.choice(preferences)))

    # GSM8K - group 2 questions
    print("Processing GSM8K...")
    # Fix: Use 'test' split for GSM8K if 'validation' is requested, as 'validation' does not exist
    gsm8k_split_key = split if split in gsm8k_data else "test" if "test" in gsm8k_data else "train"
    gsm8k_split = gsm8k_data[gsm8k_split_key]
    for i in range(0, len(gsm8k_split) - 1, 2):
        examples = [gsm8k_split[j] for j in range(i, min(i+2, len(gsm8k_split)))]
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_gsm8k(examples, True, pref))
        all_prompts.append(create_multi_question_prompt_gsm8k(examples, False, random.choice(preferences)))

    # OpenBookQA - group 3 questions
    print("Processing OpenBookQA...")
    openbook_split = openbook_data[split]
    for i in range(0, len(openbook_split) - 2, 3):
        examples = [openbook_split[j] for j in range(i, min(i+3, len(openbook_split)))]
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_openbook(examples, True, pref))
        all_prompts.append(create_multi_question_prompt_openbook(examples, False, random.choice(preferences)))

    print("Processing ELI5...")
    for i in range(min(len(eli5_dataset_slice), 5000)):
        pref = random.choice(preferences)
        all_prompts.append(create_multi_question_prompt_eli5(eli5_dataset_slice[i], True, pref))
        all_prompts.append(create_multi_question_prompt_eli5(eli5_dataset_slice[i], False, random.choice(preferences)))

    random.shuffle(all_prompts)
    return all_prompts

# Prepare training data
print("Preparing training data...")
train_prompts = prepare_combined_dataset(sciq, gsm8k, openbookqa, eli5_data[:10000], "train")
print(f"\nPreparing validation data...")
val_prompts = prepare_combined_dataset(sciq, gsm8k, openbookqa, eli5_data[10000:11000], "validation")

print(f"\n{'='*80}")
print(f"Training examples: {len(train_prompts)}")
print(f"Validation examples: {len(val_prompts)}")
print(f"{'='*80}")

train_dataset = Dataset.from_dict({"text": train_prompts})
val_dataset = Dataset.from_dict({"text": val_prompts})
print("\nReady!")

Preparing training data...
Processing SciQ...
Processing GSM8K...
Processing OpenBookQA...
Processing ELI5...

Preparing validation data...
Processing SciQ...
Processing GSM8K...
Processing OpenBookQA...
Processing ELI5...

Training examples: 28562
Validation examples: 4316

Ready!


In [ ]:
print(eli5_df.columns)

Index(['question_id', 'question', 'answers', 'ctxs'], dtype='object')


## Load Model

In [ ]:
model_name = "mistralai/Mistral-7B-v0.1"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Model loaded!")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Model loaded!


## Configure LoRA

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = prepare_model_for_kbit_training(model)
print("LoRA configured!")

LoRA configured!


## Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./adaptive-learning-3dataset",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    logging_steps=50,
    warmup_steps=200,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    save_total_limit=2,
    load_best_model_at_end=True,
    dataloader_num_workers=2,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    args=training_args,
)

print(f"Trainer ready!")
print(f"Estimated time on A100: 3-4 hours")

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/28562 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/28562 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/28562 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/4316 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4316 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/4316 [00:00<?, ? examples/s]

Trainer ready!
Estimated time on A100: 3-4 hours


In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive')

# Train
print("Starting training...\n")
trainer.train()
print("\nDone!")

Mounted at /content/drive
Starting training...



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
500,0.898600,0.861215
1000,0.857100,0.858312
1500,0.827000,0.855373
2000,0.716600,0.880738
2500,0.692500,0.884694
3000,0.682400,0.887791
3500,0.668500,0.889018


wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run
wandb: WARNING URL not available in offline run



Done!


## Save

In [ ]:
trainer.model.save_pretrained("./adaptive-learning-3dataset-final")
tokenizer.save_pretrained("./adaptive-learning-3dataset-final")
!cp -r ./adaptive-learning-3dataset-final /content/drive/MyDrive/
print("Saved!")

Saved!


## Test

In [ ]:
# Test Science
test_sci = [sciq['test'][i] for i in range(2)]
prompt = create_multi_question_prompt_sciq(test_sci, True, "videos")
test_input = prompt.split("### Evaluation:")[0] + "### Evaluation:"

inputs = tokenizer(test_input, return_tensors="pt").to("cuda")
outputs = trainer.model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print("SCIENCE TEST:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
print("\n" + "="*80 + "\n")

# Test Math
test_math = [gsm8k['test'][i] for i in range(2)]
prompt = create_multi_question_prompt_gsm8k(test_math, False, "articles")
test_input = prompt.split("### Evaluation:")[0] + "### Evaluation:"

inputs = tokenizer(test_input, return_tensors="pt").to("cuda")
outputs = trainer.model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print("MATH TEST:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


SCIENCE TEST:
### Domain: [SCIENCE]
### Context:
Oxidants and Reductants Compounds that are capable of accepting electrons, such as O 2 or F2, are calledoxidants (or oxidizing agents) because they can oxidize other compounds. In the process of accepting electrons, an oxidant is reduced. Compounds that are capable of donating electrons, such as sodium metal or cyclohexane (C6H12), are calledreductants (or reducing agents) because they can cause the reduction of another compound. In the process of donating electrons, a reductant is oxidized. These relationships are summarized in Equation 3.30: Equation 3.30 Saylor URL: http://www. saylor. org/books.

### Assessment:
Q1: Compounds that are capable of accepting electrons, such as o 2 or f2, are called what?
Student Answer: oxidants

Q2: What term in biotechnology means a genetically exact copy of an organism?
Student Answer: clone


### User Preference: videos

### Evaluation:
The student demonstrated strong understanding across all 3 ques